# Dispersion Optimizer — API Demo

Build optimal dispersion baskets with the genetic-algorithm optimizer behind the
Streamlit app, through the public `optimize()` API.

**Covered here:** vol swap · corridor · cross-corridor · long-only · vega/axe
recycling mode · bucket (group) constraints · forced/excluded names · missing-data
policies · optional score metrics · seed stability · run-bundle save & exact replay.

> Requires a Bloomberg session. Edit the **Inputs** cell and re-run top-to-bottom.

## 0. Setup

Locates the `functions` package (set `GAIA_REPO` to your repo root if not found),
imports the API, defines display helpers.

In [ ]:
import os, sys
from pathlib import Path

def _find_repo_root() -> Path:
    candidates = [os.environ.get("GAIA_REPO"), os.path.abspath("../.."), os.getcwd(),
                  os.path.abspath(".."), os.path.expanduser("~/Disp")]
    for c in candidates:
        if c and (Path(c) / "functions" / "dispersion").is_dir():
            return Path(c)
    raise FileNotFoundError(
        "Could not locate the 'functions/dispersion' package. "
        "Set GAIA_REPO to your repo root.")

sys.path.insert(0, str(_find_repo_root()))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date

from functions.dispersion import (optimize, DispersionConfig, OptimizationConstraints,
                                  get_n_exp_from_date)
from functions.dispersion.models import ProductType, MissingDataPolicy, BucketConstraint
from functions.dispersion._charts import plot_main_backtest  # internal — may change without notice

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


def show_result(result, title=""):
    """Basket + key diagnostics + backtest chart for an OptimizationResult."""
    if title:
        print(f"── {title} " + "─" * max(0, 60 - len(title)))
    if not result.long_basket:
        print("No valid solution — see debug info (rejection reasons).")
        return
    print(f"score {result.score:.4f}   converged {result.converged}   gens {result.generations_run}   "
          f"net_strike {result.net_strike * 100:.2f}%")
    long_tbl = pd.DataFrame(result.long_basket, columns=["Ticker", "Weight"])
    long_tbl["Weight %"] = (long_tbl["Weight"] * 100).round(2)
    display(long_tbl[["Ticker", "Weight %"]])
    if result.short_basket:
        print("Short:", result.short_basket)
    if result.backtest is not None and not result.backtest.timeseries.empty:
        fig = plot_main_backtest(result.backtest.timeseries,
                                 is_cross_corridor=result.is_cross_corridor)
        fig.update_layout(height=320, margin=dict(t=40, b=30, l=50, r=20), title=title or None)
        fig.show()

print("Setup OK — repo:", _find_repo_root())

## 1. Inputs

UI-style candidate tables. `Min/Max Weight` are per-name bounds in **percent of the
long side** (weights sum to 100%). The short table here is the index hedge.

In [ ]:
# ── Long candidates (edit me) ──
long_df = pd.DataFrame([
    {"Underlying": "TSLA US Equity", "Strike (%)": 50.03, "Min Weight": 5, "Max Weight": 35},
    {"Underlying": "NVDA US Equity", "Strike (%)": 48.12, "Min Weight": 5, "Max Weight": 35},
    {"Underlying": "META US Equity", "Strike (%)": 38.45, "Min Weight": 5, "Max Weight": 35},
    {"Underlying": "AMZN US Equity", "Strike (%)": 35.20, "Min Weight": 5, "Max Weight": 35},
    {"Underlying": "MSFT US Equity", "Strike (%)": 28.90, "Min Weight": 5, "Max Weight": 35},
    {"Underlying": "AAPL US Equity", "Strike (%)": 27.50, "Min Weight": 5, "Max Weight": 35},
    {"Underlying": "GOOG US Equity", "Strike (%)": 31.80, "Min Weight": 5, "Max Weight": 35},
])

# ── Short candidates (edit me) ──
short_df = pd.DataFrame([
    {"Underlying": "SPX Index", "Strike (%)": 20.00, "Min Weight": 100, "Max Weight": 100},
])

# ── Dates, tenor, constraints, score weights (edit me) ──
start_date    = date(2020, 1, 1)
maturity_date = date(2027, 7, 9)

# Canonical API columns
long_api  = long_df.rename(columns={"Underlying": "Variance Asset", "Strike (%)": "Strike Mono Var Swap (%)"})
short_api = short_df.rename(columns={"Underlying": "Variance Asset", "Strike (%)": "Strike Mono Var Swap (%)"})

n_exp = get_n_exp_from_date(maturity_date, long_df["Underlying"].tolist())
print(f"Maturity {maturity_date} -> n_exp = {n_exp} trading days")

constraints = OptimizationConstraints(
    min_stocks_long=3, max_stocks_long=6,
    min_stocks_short=1, max_stocks_short=1,
    max_net_strike=0.30,          # |weighted long strike − short strike| limit
)

score_weights = {                  # auto-normalized; 0 = inactive
    "last_carry": 0.30,            # recent payoffs
    "mean_payoff": 0.30,           # average payoff
    "hit_ratio": 0.30,             # fraction of positive days
    "min_payoff": 0.10,            # worst-day floor
}

## 2. Vol Swap — the Base Case

Long single-name vol, short index vol. This is the reference run for the sections
below: same candidates, only the config/mode changes.

In [ ]:
config_vs = DispersionConfig(
    product_type=ProductType.VOL_SWAP,
    cross_corridor=False,
    n_exp=n_exp,
    local_cap=2.5,
    missing_data_policy=MissingDataPolicy.ADAPTIVE_REWEIGHT,
)

result_vs = optimize(
    long_df=long_api, config=config_vs, constraints=constraints,
    short_df=short_api, score_weights=score_weights,
    start_date=start_date, seed=0,      # seed fixed => fully reproducible
)
show_result(result_vs, "Vol Swap — base case")

## 3. Corridor Variance Swap

Variance accrues only inside the `[70%, 130%]` corridor.

In [ ]:
config_corr = DispersionConfig(
    product_type=ProductType.VAR_SWAP_CORRIDOR,
    cross_corridor=False,
    n_exp=n_exp,
    barrier_up=1.30, barrier_down=0.70,
    local_cap=2.5,
    missing_data_policy=MissingDataPolicy.ADAPTIVE_REWEIGHT,
)

result_corr = optimize(
    long_df=long_api, config=config_corr, constraints=constraints,
    short_df=short_api, score_weights=score_weights,
    start_date=start_date, seed=0,
)
show_result(result_corr, "Corridor Var Swap")

## 4. Cross-Corridor

Each long leg = mono var swap on the **stock** − corridor swap on the **index**
inside the stock's corridor. `Variance Asset` is the shared index; strikes are per-leg.

In [ ]:
long_xc = pd.DataFrame([
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "TSLA US Equity", "Strike Cross Corridor (%)": 19.24, "Strike Mono Var Swap (%)": 50.03, "Min Weight": 5, "Max Weight": 35},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "NVDA US Equity", "Strike Cross Corridor (%)": 18.80, "Strike Mono Var Swap (%)": 48.12, "Min Weight": 5, "Max Weight": 35},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "META US Equity", "Strike Cross Corridor (%)": 20.10, "Strike Mono Var Swap (%)": 38.45, "Min Weight": 5, "Max Weight": 35},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "AMZN US Equity", "Strike Cross Corridor (%)": 19.55, "Strike Mono Var Swap (%)": 35.20, "Min Weight": 5, "Max Weight": 35},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "MSFT US Equity", "Strike Cross Corridor (%)": 21.20, "Strike Mono Var Swap (%)": 28.90, "Min Weight": 5, "Max Weight": 35},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "AAPL US Equity", "Strike Cross Corridor (%)": 21.40, "Strike Mono Var Swap (%)": 27.50, "Min Weight": 5, "Max Weight": 35},
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "GOOG US Equity", "Strike Cross Corridor (%)": 20.60, "Strike Mono Var Swap (%)": 31.80, "Min Weight": 5, "Max Weight": 35},
])
short_xc = pd.DataFrame([
    {"Variance Asset": "SPX Index", "Corridor Condition Asset": "SPX Index", "Strike Cross Corridor (%)": 20.00, "Strike Mono Var Swap (%)": 20.00, "Min Weight": 100, "Max Weight": 100},
])

config_xc = DispersionConfig(
    product_type=ProductType.VAR_SWAP_CORRIDOR,
    cross_corridor=True,
    n_exp=n_exp,
    barrier_up=1.30, barrier_down=0.70,
    local_cap=2.5,
    missing_data_policy=MissingDataPolicy.ADAPTIVE_REWEIGHT,
)
constraints_xc = OptimizationConstraints(
    min_stocks_long=3, max_stocks_long=6, min_stocks_short=1, max_stocks_short=1,
    max_net_strike=0.30,
)

result_xc = optimize(
    long_df=long_xc, config=config_xc, constraints=constraints_xc,
    short_df=short_xc, score_weights=score_weights,
    start_date=start_date, seed=0,
)
show_result(result_xc, "Cross-Corridor")

# Per-leg index/stock pairing of the delivered basket
if result_xc.long_cross_corridor:
    display(pd.DataFrame(result_xc.long_cross_corridor,
                         columns=["Stock", "Index", "Cross Strike"]).assign(
        **{"Cross Strike %": lambda d: (d["Cross Strike"] * 100).round(2)}))

## 5. Long-Only

No short side at all (`short_df=None`, `min/max_stocks_short=0`).

In [ ]:
constraints_lo = OptimizationConstraints(
    min_stocks_long=3, max_stocks_long=6,
    min_stocks_short=0, max_stocks_short=0,
    max_net_strike=0.30,
)

result_lo = optimize(
    long_df=long_api, config=config_vs, constraints=constraints_lo,
    short_df=None, score_weights=score_weights,
    start_date=start_date, seed=0,
)
show_result(result_lo, "Vol Swap — long-only")

## 6. Vega / Axe-Recycling Mode

Absolute-vega sizing: the basket total vega V is free in `[v_min, v_max]`, each name
holds `v_i = w_i·V`, and the solver can favour baskets whose vega **recycles your axe
inventory** (`axe_package_recycled` = Σ min(vᵢ, targetᵢ)/V).

- per-name hard cap: `Axe Cap` column (blank = no cap)
- recycling targets: `Axe Target` column
- the `VegaConfig` carries the total-V bounds

In [ ]:
from functions.dispersion.models import VegaConfig

long_vega = long_api.copy()
long_vega["Axe Cap"] = [40, 40, 40, 40, 40, 40, 40]          # per-name absolute vega cap
long_vega["Axe Target"] = [25, 20, 15, 0, 0, 0, 0]           # recycling inventory (0 = none)

weights_vega = dict(score_weights, axe_package_recycled=0.30)  # recycling priority

result_vega = optimize(
    long_df=long_vega, config=config_vs, constraints=constraints,
    short_df=short_api, score_weights=weights_vega,
    start_date=start_date, seed=0,
    vega=VegaConfig(v_min=50.0, v_max=200.0),                # total basket vega bounds
)
show_result(result_vega, "Vega / axe-recycling mode")

if result_vega.vega_basket:
    print(f"Total vega V = {result_vega.total_vega:.1f}   "
          f"book cleaned {result_vega.axe_cleaned if result_vega.axe_cleaned is not None else float('nan'):.1%}   "
          f"package recycled {result_vega.axe_recycled:.1%}")
    display(pd.DataFrame(result_vega.vega_basket, columns=["Ticker", "Vega"]))

## 7. Bucket (Group) Constraints

Cap the number of names and/or the total weight per group (e.g. region/sector).
Group membership comes from the `Sector` column.

In [ ]:
long_bkt = long_api.copy()
long_bkt["Sector"] = ["US", "US", "US", "US", "US", "US", "US"]   # demo: single bucket
# Example with two buckets would be: ["US","US","US","EU","EU","US","EU"]

result_bkt = optimize(
    long_df=long_bkt, config=config_vs, constraints=constraints,
    short_df=short_api, score_weights=score_weights,
    start_date=start_date, seed=0,
    bucket_constraints=[
        BucketConstraint(bucket="US", max_names=4, max_weight=0.70),
    ],
)
show_result(result_bkt, "Bucket constraints (US: ≤4 names, ≤70%)")

## 8. Forced / Excluded Names & the 0%-HR Filter

**Precedence:** excluded names are always removed → 0%-HR filter → forced names are
restored (**forced beats the filter**). Axe-target names are *not* protected from the
filter — force them if you need them in.

In [ ]:
result_filtered = optimize(
    long_df=long_api, config=config_vs, constraints=constraints,
    short_df=short_api, score_weights=score_weights,
    start_date=start_date, seed=0,
    filter_zero_hr=True,                    # drop 0%-HR longs / 100%-HR shorts
    forced_tickers=["AAPL US Equity"],      # must appear in the basket
    excluded_tickers=["TSLA US Equity"],    # never considered
)
basket = [k for k, _ in result_filtered.long_basket]
print("Basket:", basket)
assert "AAPL US Equity" in basket and "TSLA US Equity" not in basket
show_result(result_filtered, "forced=AAPL, excluded=TSLA, 0%-HR filter ON")

## 9. Missing-Data Policies

Same run under the three policies — how gap days are treated (see Backtester demo §6
for the semantics table).

In [ ]:
rows = []
for pol in [MissingDataPolicy.ADAPTIVE_REWEIGHT,
            MissingDataPolicy.FILL_ZERO,
            MissingDataPolicy.DROP_INCOMPLETE_DAYS]:
    cfg = DispersionConfig(
        product_type=ProductType.VOL_SWAP, cross_corridor=False, n_exp=n_exp,
        local_cap=2.5, missing_data_policy=pol)
    r = optimize(long_df=long_api, config=cfg, constraints=constraints,
                 short_df=short_api, score_weights=score_weights,
                 start_date=start_date, seed=0)
    rows.append({"policy": pol.name, "score": round(r.score, 4),
                 "basket": [k for k, _ in r.long_basket]})
display(pd.DataFrame(rows))

## 10. Optional Score Metrics

Any of `max_drawdown`, `cvar_5`, `sharpe_payoff`, `weighted_strike` can be added to
the blend (weight > 0). The 4 core metrics above stay the backbone.

In [ ]:
weights_tail = {
    "last_carry": 0.20, "mean_payoff": 0.20, "hit_ratio": 0.20, "min_payoff": 0.10,
    "cvar_5": 0.15,            # mean of the 5% worst days (tail protection)
    "max_drawdown": 0.15,      # equity-curve peak-to-trough (penalized)
}

result_tail = optimize(
    long_df=long_api, config=config_vs, constraints=constraints,
    short_df=short_api, score_weights=weights_tail,
    start_date=start_date, seed=0,
)
show_result(result_tail, "With tail metrics (CVaR + MaxDD)")

## 11. Seed Stability

Same config, different seeds — a healthy objective should deliver close baskets
(identical for small universes, where the local search is exhaustive).

In [ ]:
rows = []
for s in range(3):
    r = optimize(long_df=long_api, config=config_vs, constraints=constraints,
                 short_df=short_api, score_weights=score_weights,
                 start_date=start_date, seed=s)
    rows.append({"seed": s, "score": round(r.score, 5), "gens": r.generations_run,
                 "basket": [k for k, _ in r.long_basket]})
display(pd.DataFrame(rows))

## 12. Save & Replay a Run Bundle

A run bundle freezes everything (matrix, candidates, config, seed) so the run can be
**replayed exactly** later — audit trail, regression gates, sharing.

In [ ]:
from functions.dispersion.run_bundle import load_run_bundle

# Save: optimize(..., save_bundle_path="my_run_bundle") writes the bundle directory.
# Replay: same bundle + same seed => same basket, every time.
MY_BUNDLE = "my_run_bundle"

result_saved = optimize(
    long_df=long_api, config=config_vs, constraints=constraints,
    short_df=short_api, score_weights=score_weights,
    start_date=start_date, seed=0,
    save_bundle_path=MY_BUNDLE,
)

replayed = load_run_bundle(MY_BUNDLE).replay()
same = [k for k, _ in replayed.long_basket] == [k for k, _ in result_saved.long_basket]
print("Replayed basket:", replayed.long_basket)
print("Identical to the saved run:", same)

---
## Conventions

| Topic | Rule |
|-------|------|
| Reproducibility | fixed `seed` ⇒ identical run (deterministic GA + solver) |
| Delivered weights | satisfy all constraints (sum = 1, per-name bounds, strike cap, buckets) or the run says why not |
| Scored = delivered | the optimizer scores the same curve the backtest delivers (same window, same warm-up) |
| `max_net_strike` | two-sided: `abs(long_strike − short_strike) ≤ max` |
| Smoothing | interactive UI only — `optimize()` returns unsmoothed weights |
| Time budget | `time_limit_seconds` caps the GA; local search caps itself and reports |
| Prep cache | `cache_prep=True` reuses data within the same day for fast re-runs |